# PDF Parser (new) Nexis Texts for Single Files

#### This script is useful for Nexis downloads where all articles are in a single file

In [ ]:
import os
import re
import fitz  # PyMuPDF
import pytesseract
import pandas as pd
from datetime import datetime
from PIL import Image
import PyPDF2

folder_path = r"path_to_your_pdf_folder" #set location of file 
output_csv = "extracted_data.csv" #set name of output file
outlet = "Example News Outlet" # set outlet name

In [3]:
# Check if folder is accessible
print("Exists:", os.path.exists(folder_path))
print("Is Directory:", os.path.isdir(folder_path))

# List first few files
try:
    files = os.listdir(folder_path)
    print(f"Total files detected: {len(files)}")
    print("First 10 files:", files[:10])
except Exception as e:
    print(f"Error listing files: {e}")

pdf_files = [f for f in os.listdir(folder_path) if f.endswith(('.pdf', '.PDF'))]

print(f"Total PDFs found after fix: {len(pdf_files)}")
print("First 10 PDFs:", pdf_files[:10])


Exists: True
Is Directory: True
Total files detected: 11
First 10 files: ['2015.PDF', '2016.PDF', '2017.PDF', '2018.PDF', '2019.PDF', '2020.PDF', '2021.PDF', '2022.PDF', '2023.PDF', '2024.PDF']
Total PDFs found after fix: 11
First 10 PDFs: ['2015.PDF', '2016.PDF', '2017.PDF', '2018.PDF', '2019.PDF', '2020.PDF', '2021.PDF', '2022.PDF', '2023.PDF', '2024.PDF']


In [4]:
#function to parse articles starting at a specific page in the pdf - in single file PDF including multipe articles from Nexis, that page 20 but double check per 
# outlet and adjust accordingly
def extract_articles_from_pdf(pdf_path, start_page=0):
    """Extract articles starting from the given page in the PDF file."""
    articles = []
    with fitz.open(pdf_path) as doc:
        for page_num in range(start_page, len(doc)):
            page_text = doc[page_num].get_text()
            if "Body" in page_text:  #check if the page contains an article
                articles.append(page_text)
    return articles

# Display first article for verification
for filename in os.listdir(folder_path):
    if filename.endswith(('.pdf', '.PDF')):
        pdf_path = os.path.join(folder_path, filename)
        print(f"Processing file: {pdf_path}")
        text = extract_articles_from_pdf(pdf_path)
        print(text[0][:500])  # Print first 500 characters of the first article
    else:
        print(f"Skipping non-PDF file: {filename}")      


Processing file: C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\news\Parool\2015.PDF
Page 1 of 2
Chatbot Emma glimlacht na een compliment
Chatbot Emma glimlacht na een compliment
Het Parool
23 oktober 2015 vrijdag
Copyright 2015 De Persgroep Nederland BV All Rights Reserved
Section: Economie; Blz. 17
Length: 336 words
Byline: BART VAN ZOELEN
Body
'Dag Denise, hoe oud ben jij?" vragen we Denise van de Nederlandse Energiemaatschappij (NLE). "Vrouwen 
praten niet graag over hun leeftijd en ik ben daar geen uitzondering op," antwoordt ze gevat. 
Vijf procent van de vragen aan virtue
Processing file: C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\news\Parool\2016.PDF
Page 1 of 3
Groetjes van Anne van Zalando
Groetjes van Anne van Zalando
Het Parool
27 augustus 2016 zaterdag
Copyright 2016 De Persgroep Nederland BV All Rights Reserved
Section: PS Adam; Blz. 50, 51
Length: 1273 words
Byline: ANOUK VLEUGELS
Highlight: In je inbox vechten Anne v

In [5]:
dutch_to_english_months = {
    "januari": "January", "februari": "February", "maart": "March", "april": "April",
    "mei": "May", "juni": "June", "juli": "July", "augustus": "August",
    "september": "September", "oktober": "October", "november": "November", "december": "December"
}

In [6]:
def replace_dutch_months(date_string):
    """Replace Dutch month names with English equivalents."""
    for dutch, english in dutch_to_english_months.items():
        if dutch in date_string.lower():
            return date_string.lower().replace(dutch, english)
    return date_string

def clean_text(text):
    """Clean unwanted characters from text."""
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def find_date_in_text(text):
    """Find and extract date from the entire text, ignoring time information."""
    date_patterns = [
        re.compile(r'(\d{1,2})\s+(januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+(\d{4})', re.IGNORECASE),
        re.compile(r'(\d{1,2})\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})', re.IGNORECASE),
        re.compile(r'(\d{1,2})[-/](\d{1,2})[-/](\d{4})'),
        re.compile(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}', re.IGNORECASE),
    ]
    
    text = clean_text(text)
    for pattern in date_patterns:
        match = pattern.search(text)
        if match:
            return match.group().strip()
    return None

def find_authors(lines):
    """
    Extract authors from the text, assuming they appear in lines starting with 'Byline:'.
    """
    for line in lines:
        if line.startswith("Byline:"):
            # Remove "Byline:" and return the remaining string
            return line.replace("Byline:", "").strip()
    return "Unknown Authors"  # Default if no byline is found

def parse_date(date_string):
    """Parse a date string into a datetime object."""
    if date_string:
        date_string = replace_dutch_months(date_string)
        date_formats = ["%d %B %Y", "%d-%m-%Y", "%d/%m/%Y", "%B %d, %Y"]
        for date_format in date_formats:
            try:
                return datetime.strptime(date_string, date_format)
            except ValueError:
                continue
    return None

def find_title_and_outlet(lines):
    """
    Extract the actual headline from the text, then clean it by removing 'Het Financieele Dagblad'.
    """
    print("DEBUG: First few lines of the text:")
    print(lines[:10])

    headline = lines[1].strip() if len(lines) > 1 else "Unknown Title"
    if len(lines) > 2 and lines[2].strip():
        headline += f" {lines[2].strip()}"

    headline = headline.replace(outlet, "").strip()
    return headline, outlet

def parse_pdf_text(text):
    """Parse the text to extract title, outlet, date, authors, and body."""
    lines = text.split('\n')
    title, outlet = find_title_and_outlet(lines)
    date_string = find_date_in_text(text)
    date = parse_date(date_string) if date_string else None
    authors = find_authors(lines)

    body_start_idx = text.find("Body") + len("Body")
    body_end_idx = text.find("Graphic", body_start_idx)
    body_end_idx = text.find("Classification", body_start_idx) if body_end_idx == -1 else body_end_idx
    body_end_idx = len(text) if body_end_idx == -1 else body_end_idx
    body = text[body_start_idx:body_end_idx].strip()

    return title, outlet, date, authors, body

def parse_pdfs_in_folder(folder_path):
    """Parse all PDFs in a folder, skipping unwanted files, and create a DataFrame."""
    data = []
    missing_dates = []
    skip_files = {""}  # Set of filenames to skip

    for filename in os.listdir(folder_path):
        if filename.endswith(('.pdf', '.PDF')) and filename not in skip_files:
            path = os.path.join(folder_path, filename)
            articles = extract_articles_from_pdf(path)
            for text in articles:
                title, outlet, date, authors, body = parse_pdf_text(text)
                data.append({'title': title, 'outlet': outlet, 'date': date, 'authors': authors, 'body': body})

            if not date:
                missing_dates.append(filename)

    print(f"PDFs missing dates: {missing_dates}")
    return pd.DataFrame(data)


df = parse_pdfs_in_folder(folder_path)

print(df.head())
print(df.shape)

nats_or_nans = df.isna().any(axis=1)
rows_with_nats_or_nans = df[nats_or_nans]

print("Rows with missing values:")
print(rows_with_nats_or_nans.shape)

df_cleaned = df.dropna()
print("Cleaned DataFrame shape:")
print(df_cleaned.shape)

df_cleaned


DEBUG: First few lines of the text:
['Page 1 of 2', 'Chatbot Emma glimlacht na een compliment', 'Chatbot Emma glimlacht na een compliment', 'Het Parool', '23 oktober 2015 vrijdag', 'Copyright 2015 De Persgroep Nederland BV All Rights Reserved', 'Section: Economie; Blz. 17', 'Length: 336 words', 'Byline: BART VAN ZOELEN', 'Body']
DEBUG: First few lines of the text:
['Page 1 of 3', 'Vera en Roos hebben het druk', 'Vera en Roos hebben het druk', 'Het Parool', '23 oktober 2015 vrijdag', 'Copyright 2015 De Persgroep Nederland BV All Rights Reserved', 'Section: Economie; Blz. 17', 'Length: 1135 words', 'Byline: BART VAN ZOELEN', 'Body']
DEBUG: First few lines of the text:
['Page 1 of 3', 'Mobiel helpt bij praten over eten', 'Mobiel helpt bij praten over eten', 'Het Parool', '3 oktober 2015 zaterdag', 'Copyright 2015 De Persgroep Nederland BV All Rights Reserved', 'Section: Wetenschap; Blz. 38', 'Length: 622 words', 'Byline: ELLEN SCHLEBUSCH', 'Body']
DEBUG: First few lines of the text:
['Pag

,title,outlet,date,authors,body
0,Chatbot Emma glimlacht na een compliment Chatb...,Parool,2015-10-23,BART VAN ZOELEN,"'Dag Denise, hoe oud ben jij?"" vragen we Denis..."
1,Vera en Roos hebben het druk Vera en Roos hebb...,Parool,2015-10-23,BART VAN ZOELEN,Klantenservice: Virtuele medewerkers kunnen st...
2,Mobiel helpt bij praten over eten Mobiel helpt...,Parool,2015-10-03,ELLEN SCHLEBUSCH,Voedseldialoog: Wageningen UR brengt app met e...
3,Een echte slimme stad is eenspeelse stad Een e...,Parool,2015-05-30,Unknown Authors,Smart city: Technocratische benadering omzeilt...
4,Spinoza Lyceum populairst Spinoza Lyceum popul...,Parool,2015-07-09,LORIANNE VAN GELDER,AMSTERDAM - De populairste scholen van de stad...
...,...,...,...,...,...
2301,Bij nieuwe pandemie staat niemand paraat Bij n...,Parool,2025-03-01,JOP VAN KEMPEN EN BAS SOETENHORST,Vijf jaar later: Nederland is nu slechter voor...
2302,De Rolls-Royce onder de naaimachines De Rolls-...,Parool,2025-02-01,MARLOES DE MOOR,"""Je uploadt een fotootje en maakt er in twee t..."
2303,'De Tweede Wereldoorlog is nog niet met pensio...,Parool,2025-05-03,BJARN VAN DEN BERG,'Weet U het nog? U wel - maar ook het nageslac...
2304,'Niets in mijn jeugd heeft me voorbereid op he...,Parool,2025-03-08,MARCEL WIEGMAN,"""Het is gek,"" zegt Martin Rombouts. ""Ik was al..."


In [7]:
# Calculate word count for each row in the 'body' column
df_cleaned['word_count'] =df_cleaned['body'].apply(lambda x: len(str(x).split()))

# Display the DataFrame with the new 'word_count' column
df_cleaned.head()

C:\Users\joly-\AppData\Local\Temp\ipykernel_11988\614310843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['word_count'] =df_cleaned['body'].apply(lambda x: len(str(x).split()))


,title,outlet,date,authors,body,word_count
0,Chatbot Emma glimlacht na een compliment Chatb...,Parool,2015-10-23,BART VAN ZOELEN,"'Dag Denise, hoe oud ben jij?"" vragen we Denis...",238
1,Vera en Roos hebben het druk Vera en Roos hebb...,Parool,2015-10-23,BART VAN ZOELEN,Klantenservice: Virtuele medewerkers kunnen st...,249
2,Mobiel helpt bij praten over eten Mobiel helpt...,Parool,2015-10-03,ELLEN SCHLEBUSCH,Voedseldialoog: Wageningen UR brengt app met e...,288
3,Een echte slimme stad is eenspeelse stad Een e...,Parool,2015-05-30,Unknown Authors,Smart city: Technocratische benadering omzeilt...,270
4,Spinoza Lyceum populairst Spinoza Lyceum popul...,Parool,2015-07-09,LORIANNE VAN GELDER,AMSTERDAM - De populairste scholen van de stad...,210


In [8]:
df_cleaned.shape

(2298, 6)

In [9]:
# Ensure 'date' is treated as string before regex
df['date'] = df['date'].apply(
    lambda x: str(x) + " 00:00:00" if pd.notna(x) and not re.search(r'\d{2}:\d{2}:\d{2}', str(x)) else str(x)
)

# Convert to datetime
df['date'] = pd.to_datetime(df['date'].str.strip(), errors='coerce')

In [10]:
df['year'] = df['date'].dt.year

In [11]:
#check if there are any NaNs in the 'date' column
nan_exists = df['date'].isna().any()
print(f"Are there any NaNs in 'date'? {nan_exists}")

#count exactly how many NaNs there are
nan_count = df['date'].isna().sum()
print(f"Number of NaNs in 'date': {nan_count}")



Are there any NaNs in 'date'? True
Number of NaNs in 'date': 8


In [12]:
counts_per_year = df['year'].value_counts().sort_index()
print(counts_per_year)

year
2015.0    184
2016.0    195
2017.0    192
2018.0    192
2019.0    257
2020.0    139
2021.0    181
2022.0    152
2023.0    265
2024.0    298
2025.0    243
Name: count, dtype: int64


In [16]:
output_folder = os.path.dirname(folder_path)

#save to folder
df_cleaned.to_csv(os.path.join(output_folder, output_csv))
